# 07 — Fuzzy Decision-Activation Gate

**Role clarification**: this gate does NOT generate new routes. It selects
between the NR continuation and the already-computed SA continuation,
using severity (NR lateness) and flexibility (trigger fraction) as
inputs. It is a decision-activation / intervention-restraint layer, never
described here as a routing optimizer.

Reproduces corrected PHASE 4: calibration-only design (12 calibration
instances), threshold sweep with a PROGRAMMATIC selection rule, frozen
tau, and a single held-out evaluation (51 instances) framed as a
**post-review, calibration-only robustness assessment** -- not blind
independent validation, since these 51 instances were already examined
in the main daily-instance analysis (Notebook 10) before this fuzzy
redesign.


In [ ]:
import os, sys, json
import pandas as pd
import numpy as np
assert 'REPO_ROOT' in dir(), "Run notebook 00 first."
sys.path.insert(0, REPO_ROOT)
from src.fuzzy import FuzzyGate

with open(os.path.join(REPO_ROOT, "configs", "fuzzy_config.json")) as f:
    fuzzy_config = json.load(f)
print("Role (from configs/fuzzy_config.json):", fuzzy_config["role"])


## 7A — Calibration-only design: severity breakpoints (12 calibration instances)

In [ ]:
EXP_DIR = os.path.join(REPO_ROOT, "data_deidentified", "experiment_outputs")
calib_blocks = pd.read_csv(os.path.join(EXP_DIR, "fuzzy_calibration_blocks.csv"))
print(f"Calibration blocks loaded: {len(calib_blocks)} (expected 144 for 12 instances x 3 triggers x 4 shocks)")

p33 = np.percentile(calib_blocks['nr_lateness'], 33)
p67 = np.percentile(calib_blocks['nr_lateness'], 67)
print(f"Computed severity breakpoints (calibration-only): p33={p33:.4f}, p67={p67:.4f}")
print(f"Frozen (documented) breakpoints: p33={fuzzy_config['severity_breakpoints']['p33']}, "
      f"p67={fuzzy_config['severity_breakpoints']['p67']}")


## 7B — Threshold sweep (calibration-only) with programmatic selection

In [ ]:
TAU_CANDIDATES = [0.45] if RUN_MODE == "quick" else [0.30,0.35,0.40,0.45,0.50,0.55,0.60,0.65,0.70]

def evaluate_tau(gate, df, tau):
    activate = np.array([gate.fuzzy_score(l, t) for l, t in zip(df['nr_lateness'], df['trigger_fraction'])]) >= tau
    out_lateness = np.where(activate, df['sa_lateness'], df['nr_lateness'])
    pos_risk = df['nr_lateness'] > 0.01
    zero_risk = ~pos_risk
    always_sa_reduction = (df.loc[pos_risk,'nr_lateness'] - df.loc[pos_risk,'sa_lateness']).sum()
    fz_reduction = (df.loc[pos_risk,'nr_lateness'] - out_lateness[pos_risk]).sum()
    retention = 100*fz_reduction/always_sa_reduction if always_sa_reduction > 0 else float('nan')
    harm = (activate & (out_lateness > df['nr_lateness'].values)).sum()
    unnecessary = (activate & zero_risk).sum()
    return dict(tau=tau, activation_rate=100*activate.mean(), retention_pct=retention,
                harm_count=int(harm), unnecessary_count=int(unnecessary),
                mean_rsi=np.where(activate, df['rsi'], 1.0).mean(),
                mean_moved=np.where(activate, df['moved_stops'], 0).mean())

sweep_results = []
for tau in TAU_CANDIDATES:
    gate = FuzzyGate(p33, p67, tau)
    sweep_results.append(evaluate_tau(gate, calib_blocks, tau))
sweep_df = pd.DataFrame(sweep_results)
print(sweep_df.to_string(index=False))

results_dir = os.path.join(REPO_ROOT, "results")
os.makedirs(results_dir, exist_ok=True)
sweep_df.to_csv(os.path.join(results_dir, "fuzzy_threshold_sweep.csv"), index=False)


## Programmatic tau selection (harm=0 -> unnecessary=0 -> retention>=90% -> most restrained)

In [ ]:
step1 = sweep_df[sweep_df['harm_count'] == 0]
step2 = step1[step1['unnecessary_count'] == 0]
step3 = step2[step2['retention_pct'] >= (90 if RUN_MODE == "full" else 0)]
if len(step3) > 0:
    selected_tau = step3.loc[step3['tau'].idxmax(), 'tau']
    print(f"Selected tau = {selected_tau} (programmatically, via the documented selection rule)")
    if RUN_MODE == "full":
        print(f"Documented/frozen tau: {fuzzy_config['frozen_threshold_tau']}")
else:
    selected_tau = fuzzy_config['frozen_threshold_tau']
    print(f"[QUICK MODE] Tiny candidate grid did not exercise the full selection rule; "
          f"using the frozen tau={selected_tau} directly for the held-out demonstration below.")


## 7C — Held-out robustness evaluation (51 instances, single run, tau frozen)

**Framing** (exact wording, not to be altered): *"post-review
calibration-only fuzzy re-analysis evaluated on the 51 non-calibration
instances as a robustness assessment."* This is explicitly NOT called
independent or blind validation.


In [ ]:
holdout_fuzzy = pd.read_csv(os.path.join(EXP_DIR, "fuzzy_holdout_block_level.csv"))
print(f"Held-out blocks loaded: {len(holdout_fuzzy)} (expected 1,836 for full corrected dataset)")

n = len(holdout_fuzzy)
pos_risk = holdout_fuzzy['total_lateness_min_nr'] > 0.01
always_sa_reduction = (holdout_fuzzy.loc[pos_risk,'total_lateness_min_nr'] - holdout_fuzzy.loc[pos_risk,'total_lateness_min_sa']).sum()
fz_reduction = (holdout_fuzzy.loc[pos_risk,'total_lateness_min_nr'] - holdout_fuzzy.loc[pos_risk,'fz_lateness']).sum()
retention = 100*fz_reduction/always_sa_reduction
harm = (holdout_fuzzy['activate'] & (holdout_fuzzy['fz_lateness'] > holdout_fuzzy['total_lateness_min_nr']+1e-6)).sum()

print(f"Activation rate: {100*holdout_fuzzy['activate'].mean():.4f}%")
print(f"Retention: {retention:.4f}%")
print(f"Empirical no-harm within tested scenarios: {100*(1-harm/n):.4f}% ({harm}/{n} adverse)")


## 7D — 2,000 rate-matched random-gated control

In [ ]:
N_REPS = 50 if RUN_MODE == "quick" else 2000
rng = np.random.default_rng(2024)
activation_rate = holdout_fuzzy['activate'].mean()
fz_mean = holdout_fuzzy['fz_lateness'].mean()

draws = []
for _ in range(N_REPS):
    rand_activate = rng.random(n) < activation_rate
    out = np.where(rand_activate, holdout_fuzzy['total_lateness_min_sa'], holdout_fuzzy['total_lateness_min_nr'])
    draws.append(out.mean())
draws = np.array(draws)
n_outperform = int((draws <= fz_mean).sum())
p_finite = (n_outperform + 1) / (N_REPS + 1)

print(f"Fuzzy-Gated mean lateness: {fz_mean:.4f}")
print(f"Random-Gated (n={N_REPS}) mean: {draws.mean():.4f}, SD={draws.std():.4f}")
print(f"Random runs outperforming fuzzy: {n_outperform}/{N_REPS}")
print(f"Monte-Carlo p = ({n_outperform}+1)/({N_REPS}+1) = {p_finite:.6f}")
print("(Not reported as inferential: standardized descriptive separation = "
      f"{(draws.mean()-fz_mean)/draws.std():.2f} SD)")


## Expected outputs / integrity checks

In [ ]:
checks = {
    "calibration_blocks_loaded": len(calib_blocks) > 0,
    "sweep_ran": len(sweep_df) == len(TAU_CANDIDATES),
    "holdout_loaded": len(holdout_fuzzy) > 0,
    "retention_is_finite": not pd.isna(retention),
}
if RUN_MODE == "full":
    checks["holdout_full_scale"] = n == 1836
    checks["random_gate_full_scale"] = N_REPS == 2000

for k, v in checks.items():
    print(f"{'PASS' if v else 'FAIL'}  {k}")
NOTEBOOK_07_STATUS = "PASS" if all(checks.values()) else "FAIL"
print(f"\nNOTEBOOK 07 STATUS: {NOTEBOOK_07_STATUS}")
assert NOTEBOOK_07_STATUS == "PASS"
